# Exercise 2: Predict the extruder blockage with KNN
**Internet of Production – AI, week 1** · about 25 min · work in pairs

**Scenario:** An extruder blockage stops printer 1 for hours and ends the running job. In exercise 1 you saw that the energy per minute rises before a blockage. Now you turn this observation into a **classifier** that warns the head of pre-production.

**Task:** For every printing minute decide: *will an extruder blockage occur within the next 6 hours?* (class 1) or not (class 0). Use **K-nearest neighbours (KNN)** on three features derived from the energy consumption.

**Steps 1 to 3 are prepared** (label, features, split): run them and explain to your partner what each block does (5 min). The gaps start in step 5. Then you work on what matters: KNN, scaling and the metrics.

Every `___` is a gap.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay

DATA = Path('../../00_Data/printfarm')

# from exercise 1
def load_printer(k, data=DATA):
    d = pd.read_csv(data / f'Printer_{k}.csv', sep=';', decimal=',', low_memory=False).dropna(how='all')
    d['Timestamp'] = pd.to_datetime(d['Timestamp'], format='%d.%m.%Y %H:%M')
    d = d.set_index('Timestamp')
    d['Energy'] = d['Total_Power'].diff()
    return d

def load_errors(k, data=DATA):
    e = pd.read_csv(data / f'Printer_{k}_error.csv', sep=';').dropna(how='all')
    e['Timestamp'] = pd.to_datetime(e['Timestamp'], format='%d.%m.%Y %H:%M')
    return e

df = load_printer(1)
err = load_errors(1)
blockages = np.sort(err.loc[err['Error'] == 'Extruder blocked', 'Timestamp'].values)
print(len(df), 'minutes,', len(blockages), 'extruder blockages')

## Step 1: The label (prepared)
For every printing minute we need the **time until the next blockage**. The class is 1 if this time is between 0 and the horizon.

In [ ]:
HORIZON = 360                                   # minutes: warn 6 hours ahead
p = df[df['State'] == 'printing'].copy()
p['E'] = p['Energy'] * 1000                     # Wh per minute (if Total_Power is in kWh)

idx = p.index.values
nxt = np.searchsorted(blockages, idx)           # index of the next blockage
minutes_to_blockage = np.where(nxt < len(blockages),
                               (blockages[np.minimum(nxt, len(blockages) - 1)] - idx) / np.timedelta64(1, 'm'), np.inf)
p['y'] = ((minutes_to_blockage > 0) & (minutes_to_blockage <= HORIZON)).astype(int)
print(p['y'].value_counts())

## Step 2: Features from the energy consumption (prepared)
A single minute is too noisy (exercise 1). We use moving averages over the last printing minutes:
- `E30`: mean of the last 30 minutes (short term)
- `E120`: mean of the last 120 minutes
- `slope`: change of `E120` over the last 240 minutes (trend)

We keep every 10th minute so that neighbouring samples are not almost identical.

In [ ]:
p['E30'] = p['E'].rolling(30).mean()
p['E120'] = p['E'].rolling(120).mean()
p['slope'] = p['E120'] - p['E120'].shift(240)
features = ['E30', 'E120', 'slope']

data = p.dropna(subset=features).iloc[::10]
print(len(data), 'samples,', f"{data['y'].mean():.1%} class 1")
data[features + ['y']].describe().round(3)

## Step 3: Split by time
Train on the first 70 %, test on the last 30 %. Why not randomly?

In [ ]:
cut = int(len(data) * 0.7)
train, test = data.iloc[:cut], data.iloc[cut:]
X_train, y_train, X_test, y_test = train[features], train['y'], test[features], test['y']
print('train:', len(train), 'samples, class 1:', y_train.sum())
print('test :', len(test), 'samples, class 1:', y_test.sum())

## Step 4: Look at the feature space
Plot the test samples in the plane of `E120` and `slope`. Can you imagine a **decision boundary** between the classes? Where do the classes overlap?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(test.loc[test['y'] == 0, 'E120'], test.loc[test['y'] == 0, 'slope'], s=6, color='#009EE0', alpha=.4, label='no blockage within 6 h')
ax.scatter(test.loc[test['y'] == 1, 'E120'], test.loc[test['y'] == 1, 'slope'], s=14, color='#EE7F00', label='blockage within 6 h')
ax.set_xlabel('E120 (Wh/min)'); ax.set_ylabel('slope (Wh/min)'); ax.legend(); plt.show()

## Step 5: Baseline and KNN
KNN uses distances, so the features must be **scaled**. Try several values of `k`.

In [ ]:
def report(name, y_true, y_pred):
    print(f'{name:30s} accuracy {accuracy_score(y_true, y_pred):6.1%}   recall {recall_score(y_true, y_pred, zero_division=0):6.1%}   precision {precision_score(y_true, y_pred, zero_division=0):6.1%}')

base = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
report('baseline: never warn', y_test, base.predict(X_test))

for k in [5, 25, 75]:
    knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)).fit(X_train, y_train)
    report(f'KNN, k = {k}', y_test, knn.predict(X_test))

**Compare accuracy with recall and precision. Does KNN beat the baseline, and on which metric? How does `k` change the result?**

The baseline never warns and reaches about 96 % accuracy with 0 % recall: accuracy hides the problem because class 1 is rare (about 4 %). KNN raises accuracy only slightly (about 97 %) but finds about 55–60 % of the blockages (recall) with about 65–73 % real alarms (precision). A small `k` follows single points (noisy); a large `k` smooths, here with similar quality for k = 25 to 75.

## Step 6: Confusion matrix and warnings per blockage
How many blockages of the test period are announced by at least one warning in the 6 hours before? How many false alarms occur?

In [ ]:
best = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=25)).fit(X_train, y_train)
pred = best.predict(X_test)
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=['no blockage', 'blockage < 6 h']).plot(cmap='Blues')
plt.show()

alarms = test.assign(alarm=pred)
test_blockages = [t for t in pd.to_datetime(blockages) if t >= test.index[0]]
announced = sum(alarms.loc[t - pd.Timedelta(hours=6): t, 'alarm'].any() for t in test_blockages)
print(f'blockages in the test period: {len(test_blockages)}, announced by a warning: {announced}')
print('false alarms (sampled minutes):', int(((alarms['alarm'] == 1) & (alarms['y'] == 0)).sum()))

## Your assessment (three sentences)
Is the warning useful for the head of pre-production? What does a false alarm cost, what a missed blockage? What limits the result?

A warning up to 6 hours ahead gives time to service the nozzle between two jobs; a false alarm costs a short check, a missed blockage costs a failed job and hours of downtime, so a low precision is acceptable as long as recall is reasonable. The result is limited by the few blockages (34 in total, only a handful in the test period), by the fact that only printer 1 shows the rise, and by the noise of the energy signal. Next step: use a regression (time to blockage) and more features (temperature) in week 3.